<a href="https://colab.research.google.com/github/MGentieu/dl_project/blob/martin_nlp/starters/nlp-project-starter/nlp-project/notebooks/AG_news_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M1 — Problem Scoping & Data Validation

### Problem Statement
L’objectif de ce projet est d’entraîner un modèle de classification de texte (Deep Learning, architecture LSTM) pour identifier automatiquement la catégorie d'un article de presse parmi les 4 classes du dataset AG News.
Le modèle prendra en entrée une séquence de texte (titre + description) et devra prédire la catégorie correspondante avec la plus haute probabilité.

### Model Inputs
- **Texte brut** : Séquences de mots (Titres et descriptions de news).
- **Tokenisation & Embedding** : Le texte est converti en indices via un vocabulaire (max 30 000 mots), puis projeté dans un espace vectoriel dense (`emb_dim: 128`).
- **Séquence** : Longueur maximale fixée à 256 tokens.

### Model Outputs
- **Vecteur de probabilités** : Un vecteur de taille 4 (Softmax).
- **Classe prédite** : L'index ayant la probabilité maximale (0–3) correspondant aux labels (World, Sports, Business, Sci/Tech).

### Evaluation Metrics
- **Accuracy** : Métrique principale pour ce dataset équilibré.
- **Macro-F1 Score** : Pour évaluer la performance moyenne sur toutes les classes sans être biaisé par les classes majoritaires (bien que ce dataset soit équilibré).
- **Confusion Matrix** : Pour analyser les erreurs spécifiques (ex: confusion entre Business et Sci/Tech).

### Dataset Split
Le dataset AG News est divisé comme suit :
- **Train set** : 120 000 échantillons.
- **Test set** : 7 600 échantillons.

## Data Card – AG News Dataset

### 1. Dataset Summary
Le dataset AG News est un sous-ensemble du corpus AG d'articles de presse, construit par l'académique Xiang Zhang. Il est utilisé comme standard pour la classification de texte à grande échelle.

### 2. Composition & Classes
- **Taille totale :** 127 600 échantillons.
- **Classes (4) :**
  1. World (Monde)
  2. Sports
  3. Business (Économie)
  4. Sci/Tech (Science & Technologie)

**Distribution :** Le dataset est parfaitement équilibré avec 30 000 échantillons d'entraînement et 1 900 échantillons de test pour chaque classe.

### 3. Intended Use & Limitations
- **Usage :** Benchmarking de modèles de classification (BoW, CNNs, RNNs/LSTMs, Transformers).
- **Limitations :** Les textes sont courts (titres/snippets) et ne contiennent pas l'article complet. Le contexte temporel (années 2000) peut rendre certains termes obsolètes.

### 4. Preprocessing
Le pipeline de traitement (défini dans `src/data.py`) applique :
- **Lowercasing** : Uniformisation de la casse.
- **Truncation/Padding** : Les séquences sont tronquées ou complétées pour atteindre une longueur fixe ou dynamique par batch (max 256).

### Step 0 — Installation du projet et vérification de l'état du GPU

dans le terminal de Google Colab, exécutez la commande :

```bash
git clone https://github.com/MGentieu/dl_project.git
```


In [1]:
!nvidia-smi || echo "nvidia-smi unavailable (CPU runtime)"


Thu Dec  4 09:43:33 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 1 — Point the notebook at the project folder
This cell makes sure the notebook is executing inside the `nlp-project` directory.
If it raises a `FileNotFoundError`, double-check where you uploaded/cloned the folder, adjust the path, and rerun.

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
elif PROJECT_ROOT.name == "content":
    candidate = PROJECT_ROOT / "dl_project/starters/nlp-project-starter/nlp-project"
    if candidate.exists():
        PROJECT_ROOT = candidate.resolve()

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        f"Could not locate project root at {PROJECT_ROOT}. Upload or clone nlp-project before proceeding."
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")


Project root: /content/dl_project/starters/nlp-project-starter/nlp-project


### Step 2 — Install the project requirements
This command reads `requirements.txt` and installs the exact package versions used locally. Expect a lot of output; that's normal. If installation fails, run the cell again before moving on.

In [3]:
# Install project dependencies listed in requirements.txt
!pip install -r requirements.txt


In [4]:
import pandas as pd
import json
import glob

# M2 — Baseline Model Implementation

Nous utilisons une architecture **LSTM (Long Short-Term Memory)** bidirectionnelle comme modèle baseline.

**Architecture définie dans `nlp_agnews.yaml` :**
- Embedding Dimension : 128
- Hidden Dimension : 256
- Layers : 1
- Bidirectional : True

**Objectifs du Smoke Test :**
Avant de lancer un entraînement long, nous validons que :
1. Le dataset se télécharge et le vocabulaire se construit correctement.
2. L'architecture du modèle accepte les tenseurs d'entrée (Batch Size, Seq Len).
3. Une "forward pass" fonctionne sur un mini-batch sans erreur de dimension.

*On commence dans un premier temps par un sanity check.*

### Step 3 — Run the smoke test
This quick check downloads AG News (first run only), builds the vocabulary, and runs one mini-batch through the LSTM. It saves `outputs/smoke_metrics.json` so you know the pipeline works.
If the cell reports a network/download issue, wait a few seconds and rerun it.

In [5]:
from src import smoke_check

smoke_path = smoke_check.run_smoke("configs/nlp_agnews.yaml")
print(smoke_path.read_text())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{
  "loss": 1.3951404094696045,
  "batch_size": 64,
  "seq_len": 69,
  "num_classes": 4
}


## 1. Review smoke-test output
- Confirm the previous cell printed a JSON block (loss, batch size, seq_len).
- You should now see `outputs/smoke_metrics.json` in the file browser on the left.
- Only need a quick check? You can stop here. Ready for full training? Continue to Section 2.
- If anything failed, read the error message, fix the issue, and rerun the smoke cell before moving on.

# M3 — Optimization & Regularization

Pour assurer une bonne généralisation sur le texte, nous intégrons des stratégies d'optimisation configurées dans `configs/nlp_agnews.yaml` :

### 1. Regularization
- **Dropout :** Appliqué avec une probabilité de 0.2 pour éviter que le LSTM ne se base trop sur des neurones spécifiques.
- **Weight Decay :** Fixé à `1.0e-2` dans l'optimiseur AdamW pour pénaliser les grands poids.

### 2. Optimization
- **Optimizer :** AdamW, plus stable que SGD pour les RNNs.
- **Learning Rate :** Initialisé à `1.0e-3`.

### 3. Early Stopping
- **Patience :** 5 époques. L'entraînement s'arrête si la validation loss ne s'améliore pas, sauvegardant automatiquement le meilleur modèle (`best.pt`).


Pour répondre au jalon M4 (Ablation Studies) de manière rigoureuse et reproductible, nous avons créé le script run_ablations.py. Ce script charge la configuration de base (configs/nlp_agnews.yaml), applique des modifications dynamiques pour chaque hypothèse, et lance l'entraînement séquentiellement.

Voici le détail des 5 configurations testées :

0. Baseline,"Aucune (Bi-LSTM 256, AdamW, Dropout 0.2)",Point de référence des performances.
1. Light Architecture,"hidden_dim: 64, bidirectional: False","Vérifier si un modèle beaucoup plus léger (et rapide) suffit pour cette tâche, ou si la complexité est nécessaire."
2. High Learning Rate,lr: 0.005 (au lieu de 0.001),Tester la stabilité de l'entraînement. Un LR plus élevé peut accélérer la convergence mais risque de déstabiliser le LSTM.
3. Optimizer SGD,"optimizer: sgd, lr: 0.01, momentum: 0.9","Comparer AdamW (adaptatif) à SGD (classique). AdamW est généralement supérieur pour les RNNs, nous voulons le confirmer."
4. Heavy Regularization,"hidden_dim: 512, dropout: 0.5",Tester un modèle à très forte capacité (512 neurones) mais fortement régularisé (50% dropout) pour voir si cela améliore la généralisation.

In [6]:
!python src/run_ablations.py


🚀 Lancement de l'expérience : baseline
Epoch 1/10
Epoch 2/10
Epoch 3/10
Epoch 4/10
Epoch 5/10
Epoch 6/10
Epoch 7/10
Epoch 8/10
Epoch 9/10
Epoch 10/10
Done. Best val F1-macro: 0.9036. Checkpoint: outputs/baseline/best.pt
✅ Expérience baseline terminée avec succès.

🚀 Lancement de l'expérience : exp_1_light
Epoch 1/10
Epoch 2/10
Epoch 3/10
Epoch 4/10
Epoch 5/10
Epoch 6/10
Epoch 7/10
Epoch 8/10
Epoch 9/10
Epoch 10/10
Done. Best val F1-macro: 0.9034. Checkpoint: outputs/exp_1_light/best.pt
✅ Expérience exp_1_light terminée avec succès.

🚀 Lancement de l'expérience : exp_2_high_lr
Epoch 1/10
Epoch 2/10
Epoch 3/10
Epoch 4/10
Epoch 5/10
Epoch 6/10
Epoch 7/10
Epoch 8/10
Done. Best val F1-macro: 0.9106. Checkpoint: outputs/exp_2_high_lr/best.pt
✅ Expérience exp_2_high_lr terminée avec succès.

🚀 Lancement de l'expérience : exp_3_sgd
Epoch 1/10
Epoch 2/10
Epoch 3/10
Epoch 4/10
Epoch 5/10
Epoch 6/10
Epoch 7/10
Epoch 8/10
Epoch 9/10
Epoch 10/10
Done. Best val F1-macro: 0.8323. Checkpoint: outputs

In [7]:
# Train the model using the main configuration file. Expect visible progress bars.
# The first epochs may start slowly while the dataset finishes downloading.
#!python src/train.py --config configs/nlp_agnews.yaml


In [8]:
# Evaluate the best checkpoint produced during training.
# This prints validation/test metrics and writes eval.json to the outputs folder.
#!python src/evaluate.py --config configs/nlp_agnews.yaml --ckpt outputs/best.pt


### Step 4 — What should I see now?
#### exp_dirs :
- outputs/baseline,
- outputs/exp_1_light,
- outputs/exp_2_high_lr,
- outputs/exp_3_sgd,
- outputs/exp_4_heavy_reg.


# M4 — Ablation Studies & Analysis

Nous menons des expériences pour isoler l'impact de certains hyperparamètres.
*Note : Remplissez le tableau ci-dessous avec vos propres résultats issus de `outputs/log.csv`.*

### Expérience : Impact de la dimensionnalité (Hidden Dim)
Nous comparons la baseline avec des versions différentes de notre modèle (différences décrites précédemment.)

Nous obtenons les résultats suivants :

In [12]:
results = []

# Liste des dossiers d'expérience
exp_dirs = [
    "outputs/baseline",
    "outputs/exp_1_light",
    "outputs/exp_2_high_lr",
    "outputs/exp_3_sgd",
    "outputs/exp_4_heavy_reg"
]

for d in exp_dirs:
    exp_name = os.path.basename(d)

    # Initialisation des valeurs
    best_val_acc = "N/A"
    best_val_f1 = "N/A"

    # 1. Essayer de lire log.csv pour avoir l'historique complet (et l'accuracy)
    log_file = os.path.join(d, "log.csv")
    if os.path.exists(log_file):
        try:
            df_log = pd.read_csv(log_file)
            # On récupère le max de chaque colonne
            if "val_acc" in df_log.columns:
                best_val_acc = df_log["val_acc"].max()
            if "val_f1_macro" in df_log.columns:
                best_val_f1 = df_log["val_f1_macro"].max()
        except Exception as e:
            print(f"Erreur de lecture du log pour {exp_name}: {e}")

    # 2. Fallback sur metrics.json si log.csv est vide ou manquant (pour F1)
    if best_val_f1 == "N/A":
        metric_file = os.path.join(d, "metrics.json")
        if os.path.exists(metric_file):
            with open(metric_file) as f:
                data = json.load(f)
                best_val_f1 = data.get("best_val_f1_macro", "N/A")

    results.append({
        "Experience": exp_name,
        "Best Val Accuracy": best_val_acc,
        "Best Val F1 (Macro)": best_val_f1
    })

# Création et affichage du DataFrame
df = pd.DataFrame(results)

# Tri par F1 score décroissant
if not df.empty and "Best Val F1 (Macro)" in df.columns:
    # Conversion temporaire pour le tri (les N/A deviennent NaN)
    df["sort_col"] = pd.to_numeric(df["Best Val F1 (Macro)"], errors='coerce')
    df = df.sort_values(by="sort_col", ascending=False).drop(columns=["sort_col"])

print("=== M4: Ablation Studies Results (Enriched) ===")
display(df)

=== M4: Ablation Studies Results (Enriched) ===


,Experience,Best Val Accuracy,Best Val F1 (Macro)
2,exp_2_high_lr,0.9106,0.9106
4,exp_4_heavy_reg,0.9056,0.9051
0,baseline,0.9041,0.9036
1,exp_1_light,0.9036,0.9034
3,exp_3_sgd,0.8362,0.8323


# M5 — Reporting & Final Delivery

### 1. Métriques Globales
Le script `evaluate.py` a généré le fichier `outputs/eval.json`.
L'accuracy finale sur le jeu de test est affichée ci-dessus.

### 2. Analyse par Classe (Confusion Matrix)
L'analyse de la matrice de confusion (générée dans `outputs/`) permet de comprendre les biais résiduels.

*Points d'attention typiques sur AG News :*
- **Sci/Tech vs Business :** Confusion fréquente sur les articles parlant de rachat d'entreprises technologiques.
- **Sports :** Généralement la classe la mieux prédite grâce à un vocabulaire très spécifique.

Vous pouvez visualiser la matrice de confusion avec le code suivant :